In [1]:
################################################################################
                              # Environment setup #
################################################################################
library(Seurat)
library(ggplot2)
library(patchwork)
library(dplyr)
library(stringr)

path <- "/scratch/user/uqqvo1/Medulloblastoma_analysis/scripts/"
work_dir <- dirname(path)
work_dir <- str_split(work_dir, 'scripts/')[[1]][1]
setwd(work_dir)


data_dir <- paste0(work_dir, 'data/')

#### Constructing the paths to the spatial data #####
data_dirs <- c('data/Visium94_A1_Hybrid_treated/',
                   'data/Visium94_B1_Hybrid_treated/',
                   'data/Visium94_C1_Hybrid_untreated/',
                   'data/Visium94_D1_Hybrid_untreated/')
data_names <- c('A1_treated', 'B1_treated', 'C1_untreated', 'D1_untreated')

The legacy packages maptools, rgdal, and rgeos, underpinning the sp package,
which was just loaded, were retired in October 2023.
Please refer to R-spatial evolution reports for details, especially
https://r-spatial.org/r/2023/05/15/evolution4.html.
It may be desirable to make the sf package available;
package maintainers should consider adding sf to Suggests:.

Attaching SeuratObject


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
################################################################################
                # Using Ryan's pre-filtering of the data #
################################################################################
# Loading in IDs after filtering #
id_dir <- 'data/filter_ids/'
id_files <- list.files(id_dir)
id_files <- id_files[order(id_files)] 
ids <- list()
for (i in 1:length(id_files)) {
  ids[[i]] <- read.table(paste(id_dir, id_files[i], sep=''))[,1]
}

spatials <- list() # Creating the spatial objects #
for (i in 1:length(data_names)) {
  # Loading the spatial data #
  spatial <- Load10X_Spatial(data_dirs[i],
                             filename = "filtered_feature_bc_matrix.h5",
                             assay = "Spatial",
                             slice = "slice1",
                             filter.matrix = TRUE,
                             to.upper = FALSE,
  )
    # Performing the subset to the spots Ryan is working with #
  print("This many cells loaded from seurat: ")
  print(length(spatial$orig.ident))
  print("This many cells filtered: ")
  print(length(ids[[i]]))
  
  spatial <- subset(spatial, cells=ids[[i]])
  
  print("Length after subsetting to filtered Ids:")
  print(length(spatial$orig.ident))
    
    
    spatial=spatial[,unname(which(colSums(GetAssayData(spatial))!=0))]
    
  # SCTransforming #
  spatial <- SCTransform(spatial, assay = "Spatial", verbose = FALSE,
                         return.only.var.genes=F #Ensures all genes returned !
                         )
  
  spatials[[data_names[i]]] <- spatial
  
  saveRDS(spatial, paste('data/seurat_rds/', data_names[i], '.rds', sep=''))
}

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "This many cells loaded from seurat: "
[1] 3414
[1] "This many cells filtered: "
[1] 3414
[1] "Length after subsetting to filtered Ids:"
[1] 3414


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "This many cells loaded from seurat: "
[1] 3930
[1] "This many cells filtered: "
[1] 3930
[1] "Length after subsetting to filtered Ids:"
[1] 3930


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "This many cells loaded from seurat: "
[1] 4134
[1] "This many cells filtered: "
[1] 4134
[1] "Length after subsetting to filtered Ids:"
[1] 4134


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "This many cells loaded from seurat: "
[1] 3504
[1] "This many cells filtered: "
[1] 3504
[1] "Length after subsetting to filtered Ids:"
[1] 3504


In [3]:
################################################################################
          # Using the data by themselves, applying SCTransform #
################################################################################
# Loading in IDs after filtering #
id_dir <- 'data/filter_ids/'
id_files <- list.files(id_dir)
id_files <- id_files[order(id_files)]
ids <- list()
for (i in 1:length(id_files)) {
  ids[[i]] <- read.table(paste(id_dir, id_files[i], sep=''))[,1]
}

spatials <- list() # Creating the spatial objects #
for (i in 1:length(data_names)) {
  # Loading the spatial data #
  spatial <- Load10X_Spatial(data_dirs[i])
  
  # Performing the subset to the spots Ryan is working with #
  print("This many cells loaded from seurat: ")
  print(length(spatial$orig.ident))
  print("This many cells filtered by Ryan: ")
  print(length(ids[[i]]))
  
  spatial <- subset(spatial, cells=ids[[i]])
  
  print("Length after subsetting to Ryan Ids:")
  print(length(spatial$orig.ident))
  
  spatials[[data_names[i]]] <- spatial}

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "This many cells loaded from seurat: "
[1] 3414
[1] "This many cells filtered by Ryan: "
[1] 3414
[1] "Length after subsetting to Ryan Ids:"
[1] 3414


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "This many cells loaded from seurat: "
[1] 3930
[1] "This many cells filtered by Ryan: "
[1] 3930
[1] "Length after subsetting to Ryan Ids:"
[1] 3930


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "This many cells loaded from seurat: "
[1] 4134
[1] "This many cells filtered by Ryan: "
[1] 4134
[1] "Length after subsetting to Ryan Ids:"
[1] 4134


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "This many cells loaded from seurat: "
[1] 3504
[1] "This many cells filtered by Ryan: "
[1] 3504
[1] "Length after subsetting to Ryan Ids:"
[1] 3504


In [4]:
# Merging  #
merged <- merge(spatials[[1]], 
                y= spatials[-1], add.cell.ids=data_names)
merged=merged[,unname(which(colSums(GetAssayData(merged))!=0))]
merged <- SCTransform(merged, assay = "Spatial", verbose = T,
            return.only.var.genes=F #Ensures all genes returned !
)

# Adding meta data and saving #
name_split <- str_split(names(merged$orig.ident), '_')
sample_names <- c()
treatments <- c()
for (i in 1:length(name_split)) {
  sample_name <- paste(name_split[[i]][1:2], collapse='_')
  sample_names <- c(sample_names, sample_name)
  
  treatment <- name_split[[i]][2]
  treatments <- c(treatments, treatment)
}
merged <- AddMetaData(merged, sample_names, 'sample')
merged <- AddMetaData(merged, treatments, 'treatment')
print(unique(merged$sample))
print(unique(merged$treatment))

saveRDS(merged, 'data/seurat_rds/all.rds')

Calculating cell attributes from input UMI matrix: log_umi

Variance stabilizing transformation of count matrix of size 34448 by 14978

Model formula is y ~ log_umi

Get Negative Binomial regression parameters per gene

Using 2000 genes, 5000 cells



  |======================================================================| 100%


Found 61 outliers - those will be ignored in fitting/regularization step


Second step: Get residuals using fitted parameters for 34448 genes



  |======================================================================| 100%


Computing corrected count matrix for 34448 genes



  |======================================================================| 100%


Calculating gene attributes

Wall clock passed: Time difference of 2.750682 mins

Determine variable features

Place corrected count matrix in counts slot

Centering data matrix

Set default assay to SCT



[1] "A1_treated"   "B1_treated"   "C1_untreated" "D1_untreated"
[1] "treated"   "untreated"
